# M2M100 + LoRA 최종 그룹 분할 실험

이 노트북은 최종 3 epoch 학습 코드를 연결한다. 기본 모델·어댑터·원문·정답·예측 출력은 포함하지 않는다. 모델 다운로드와 자료 준비는 README의 별도 명령이다.

In [ ]:
from pathlib import Path
import json, sys, subprocess

# 프로젝트 루트 또는 notebooks/ 폴더에서 실행한다.
ROOT = Path.cwd().resolve()
if not (ROOT / "README.md").is_file():
    ROOT = ROOT.parent
assert (ROOT / "configs").is_dir(), "프로젝트 폴더에서 JupyterLab을 실행해야 합니다."

# 실행 플래그가 False이면 자료를 읽거나 모델을 로딩하지 않는다.
RUN_TRAINING = False
RUN_EVALUATION = False
RUN_APP = False


In [ ]:
config = json.loads((ROOT / "configs/train.json").read_text(encoding="utf-8"))
print(json.dumps(config, ensure_ascii=False, indent=2))

## 무엇을 학습하는가

기본 모델은 이미 사전학습된 M2M100 encoder–decoder다. LoRA는 기존 큰 행렬을 고정하고 작은 A와 B를 학습하여 W + (alpha/r) BA 형태의 변화를 더한다. 이 실험은 q_proj와 v_proj에 rank 8을 사용한다. 전체 485,085,184개 중 학습 대상은 1,179,648개다. PyTorch와 Transformers/PEFT가 해당 연산을 수행하며 기본 번역 지식을 처음부터 만든 실험은 아니다.

원문이나 정답이 같은 연결 그룹을 분할 사이에 나누지 않는다. 학습에 사용하지 않은 test도 기본 모델의 사전학습에서 본 적 없는 자료라고 보장할 수는 없다.

In [ ]:
print((ROOT / "train_extended.py").read_text(encoding="utf-8"))

## 명시적 학습

Token cross entropy의 PAD 자리는 -100으로 무시한다. Gradient accumulation은 마지막 작은 묶음까지 실제 정답 토큰 수로 가중한다. AMP는 일부 계산을 FP16으로 수행하고 LoRA 학습 파라미터는 FP32로 유지한다. 검증 손실 최저 어댑터를 저장한다. 아래 플래그를 True로 바꾸면 학습이 시작된다. --init-adapter는 가중치 초기화이며 완전 재개가 아니다.

In [ ]:
if RUN_TRAINING:
    subprocess.run([sys.executable, "train_extended.py", "--config", "configs/train.json", "--run-dir", "checkpoints/reference"], cwd=ROOT, check=True)
else:
    print("학습 실행 안 함")

## 평가와 CPU 앱

동일 test에서 기본 모델과 학습 어댑터를 beam 4로 평가한다. 정답은 생성 모델에 전달하지 않는다. BLEU와 chrF는 corpus 단위 지표이며 개별 문장의 정확성 보장은 아니다. CPU 앱은 같은 기본 모델과 어댑터를 로컬로 읽는다. INT8은 별도 추론 조건이며 기존 CUDA 시험 점수를 그대로 INT8 성능으로 표시할 수 없다.

In [ ]:
if RUN_EVALUATION:
    subprocess.run([sys.executable, "evaluate_extended.py", "--run-dir", "checkpoints/reference"], cwd=ROOT, check=True)
if RUN_APP:
    subprocess.Popen([sys.executable, "cpu_app.py", "--checkpoint", "checkpoints/reference/adapter"], cwd=ROOT)
print(json.dumps(json.loads((ROOT / "results/reference_metrics.json").read_text(encoding="utf-8")), ensure_ascii=False, indent=2))